## Empirical FC

In [ ]:
import matplotlib.pyplot as plt

import sys
from pathlib import Path

sys.path.append(str(Path("../..").resolve()))

from empirical_FC.import_eso import return_eso_controls, get_fc_orders, print_order_names

empirical_fc_matrix_list = return_eso_controls(printing=False)

plt.imshow(empirical_fc_matrix_list[1])

columns = empirical_fc_matrix_list[1].columns
print(f'Columns: {columns}')

print(f'Lenght of empirical fc matrix list: {len(empirical_fc_matrix_list)}')

# for matrix in matrix_list[:5]:
#     plt.imshow(matrix)
#     plt.show()


## Load in ordering

In [ ]:
orders = get_fc_orders(columns)

print_order_names()

hemisphere_order, lobar_bilateral_order, lobar_hemi_order, subcortical_first_order, functional_order = get_fc_orders(columns)

named_orders = {
    'hemisphere_order': hemisphere_order,
            'lobar_bilateral_order': lobar_bilateral_order,
            'lobar_hemi_order': lobar_hemi_order,
            'subcortical_first_order': subcortical_first_order,
            'functional_order':functional_order
}

for key in named_orders.keys():
    print(f'Order name: {key}')
    order = named_orders[key]
    plt.title(key)
    plt.imshow(empirical_fc_matrix_list[1].loc[order, order])
    plt.show()

## Order empirical FC according to hemispheric

In [ ]:
hemisphere_order = hemisphere_order

ordered_empfc_matrix_list = []

for index, matrix in enumerate(empirical_fc_matrix_list):
    # print(f'Index: {index}')

    ordered_matrix = matrix.loc[hemisphere_order, hemisphere_order]
    ordered_empfc_matrix_list.append(ordered_matrix)

for matrix in ordered_empfc_matrix_list[:3]:
    plt.title('hemisphere_order')
    plt.imshow(matrix)
    plt.show()

## Empirical structural connectivity

In [ ]:
import matplotlib.pyplot as plt

# load in SC
import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from empirical_FC.import_eso import return_eso_controls
from empirical_SC.AAL90_skocha.import_skocha import return_skocha_sc, return_skocha_sc_dictionary

empirical_sc_matrix_list = return_skocha_sc(printing=True)

# for matrix in empirical_sc_matrix_list[:4]:
#     plt.imshow(matrix)


## Order structural FC according to hemispheric

In [ ]:
hemisphere_order = hemisphere_order

ordered_emp_sc_matrix_list = []

for index, matrix in enumerate(empirical_sc_matrix_list):
    # print(f'Index: {index}')

    ordered_matrix = matrix.loc[hemisphere_order, hemisphere_order]
    ordered_emp_sc_matrix_list.append(ordered_matrix)

# for matrix in ordered_emp_sc_matrix_list[:3]:
#     plt.title('hemisphere_order')
#     plt.imshow(matrix)
#     plt.show()

## Check that ordering is identical:

In [ ]:
assert (ordered_empfc_matrix_list[1].columns == ordered_emp_sc_matrix_list[1].columns).all()

## Generate average structural connectome

In [ ]:
import numpy as np

# Average SC across subjects

sc_matrices = ordered_emp_sc_matrix_list

mean_emp_sc = np.mean(sc_matrices, axis=0)
region_names = sc_matrices[1].columns

plt.title('average structural connectome')
plt.imshow(mean_emp_sc)
plt.show()

## Generate average functional connectome

In [ ]:
fc_matrices = ordered_empfc_matrix_list

# Average empirical FC across subjects
mean_emp_fc = np.mean(fc_matrices, axis=0)

plt.title('average empirical FC')
plt.imshow(mean_emp_fc)
plt.show()

## OU process

In [ ]:

def ou_fc(connectivity, coupling, sigma=1.0):
    """
    Analytical OU FC from Saggio et al. (2016).

    Parameters
    ----------
    connectivity : array-like, shape (N, N)
        Structural connectivity matrix W.

    coupling : float
        Global coupling parameter c.

    sigma : float
        Gaussian noise amplitude.
        Does not affect the resulting Pearson FC because
        it only scales the covariance.

    Returns
    -------
    fc : ndarray, shape (N, N)
        Analytical functional connectivity matrix.

    cov : ndarray, shape (N, N)
        Stationary covariance matrix.
    """

    W = np.asarray(connectivity, dtype=float).copy()

    # Saggio explicitly removes self-connections
    np.fill_diagonal(W, 0)

    N = W.shape[0]

    # A = -I + cW
    A = -np.eye(N) + coupling * W

    # stationary covariance for symmetric W
    # C = -(sigma^2 / 2) A^-1
    cov = -(sigma**2 / 2) * np.linalg.inv(A)

    # covariance -> Pearson correlation
    std = np.sqrt(np.diag(cov))

    fc = cov / np.outer(std, std)

    return fc, cov

def get_critical_coupling_ou(connectivity):

    W = np.asarray(connectivity, dtype=float).copy()

    np.fill_diagonal(W, 0)

    # symmetric SC -> use eigvalsh
    eigenvalues = np.linalg.eigvalsh(W)

    lambda_max = np.max(eigenvalues)

    return 1.0 / lambda_max

## Get critical coupling

In [ ]:
crit = get_critical_coupling_ou(mean_emp_sc)

print(f"Critical coupling: {crit}")

## Parameter sweep:

In [ ]:
import pandas as pd

couplings = np.linspace(
    0.001,
    0.9999,
    1000
) * crit


emp_fc = np.asarray(mean_emp_fc, dtype=float).copy()

# empirical FC has NaN self-correlations
np.fill_diagonal(emp_fc, 1.0)

print("NaNs in empirical FC:", np.isnan(emp_fc).sum())
# should print 0

log = pd.DataFrame(columns=[
    'c',
    'c_over_crit',
    'corr'
])

print("NaNs in empirical FC:", np.isnan(emp_fc).sum())
print("NaNs in SC:", np.isnan(np.asarray(mean_emp_sc, dtype=float)).sum())

test_fc, test_cov = ou_fc(
    mean_emp_sc,
    coupling=couplings[0],
    sigma=1.0
)

print("NaNs in simulated covariance:", np.isnan(test_cov).sum())
print("NaNs in simulated FC:", np.isnan(test_fc).sum())

print("Cov diagonal min/max:",
      np.diag(test_cov).min(),
      np.diag(test_cov).max())


for coupling in couplings:

    sim_fc, sim_cov = ou_fc(
        mean_emp_sc,
        coupling=coupling,
        sigma=1.0
    )

    # --------------------------------------------------------
    # Saggio parameter exploration:
    # retain diagonal during search for best c
    # --------------------------------------------------------

    tri_with_diag = np.tril_indices_from(
        emp_fc,
        k=0
    )

    corr = np.corrcoef(
        sim_fc[tri_with_diag],
        emp_fc[tri_with_diag]
    )[0, 1]

    log.loc[len(log)] = {
        'c': coupling,
        'c_over_crit': coupling / crit,
        'corr': corr
    }



max_idx = log['corr'].idxmax()

best_c = log.loc[max_idx, 'c']
best_corr_sweep = log.loc[max_idx, 'corr']

print(f"Critical coupling: {crit}")
print(f"Best coupling:     {best_c}")
print(f"c / c_crit:        {best_c / crit}")
print(f"Sweep correlation: {best_corr_sweep}")



## ! recompute corr with removed diagonals:

In [ ]:
best_fc, best_cov = ou_fc(
    mean_emp_sc,
    coupling=best_c
)

tri_no_diag = np.tril_indices_from(
    emp_fc,
    k=-1
)

final_corr = np.corrcoef(
    best_fc[tri_no_diag],
    emp_fc[tri_no_diag]
)[0, 1]

print(f"Final off-diagonal correlation: {final_corr}")

## Plotting:

In [ ]:
# plt.figure(figsize=(4, 6))

plt.plot(
    log['c'],
    log['corr']
)

plt.axvline(
    crit,
    linestyle='--',
    label=fr'$c_{{critic}}$ = {crit:.3f}'
)

plt.scatter(
    best_c,
    best_corr_sweep,
    color='red',
    zorder=5
)

plt.axvline(
    best_c,
    color='red',
    linestyle='--',
    label=fr'$c_{{best}}$ = {best_c:.3f}'
)

# same visual range as paper
plt.xlim(0, 1.5)
plt.ylim(0.35, 0.75)

plt.xlabel('Global coupling parameter c')
plt.ylabel('Correlation between empirical FC and aFC')

plt.legend()
plt.show()

## Narrow plotting:

In [ ]:
plt.figure(figsize=(4, 6))

plt.plot(
    log['c'],
    log['corr']
)

plt.axvline(
    crit,
    linestyle='--',
    label=fr'$c_{{critic}}$ = {crit:.3f}'
)

plt.scatter(
    best_c,
    best_corr_sweep,
    color='red',
    zorder=5
)

plt.axvline(
    best_c,
    color='red',
    linestyle='--',
    label=fr'$c_{{best}}$ = {best_c:.3f}'
)

# same visual range as paper
plt.xlim(0, 1.5)
plt.ylim(0.35, 0.75)

plt.xlabel('Global coupling parameter c')
plt.ylabel('Correlation between empirical FC and aFC')

plt.legend()
plt.show()

## Rerun with skewed SC and skewed FC matrix (including sinuses)

In [ ]:
# regions you want at the OUTSIDE of the matrix
leading_left = [
    'L Superior Orbital Gyrus',       # Frontal_Sup_Orb_L
    'L Middle Orbital Gyrus',         # Frontal_Mid_Orb_L
    'L IFG (p. Orbitalis)',           # Frontal_Inf_Orb_L

    # add the exact names for these from your region_names:
    # Frontal_Med_Orb_L
    # Rectus_L
    'L Olfactory cortex',
    # Temporal_Pole_Sup_L
    # Temporal_Pole_Mid_L
    # ParaHippocampal_L
]

# all left hemisphere regions
all_left = [
    r for r in region_names
    if str(r).startswith('L ')
]

# leading regions first, then everything else
left_order = (
    leading_left +
    [r for r in all_left if r not in leading_left]
)

# create right hemisphere in REVERSE order
right_lookup = {
    r[2:]: r
    for r in region_names
    if str(r).startswith('R ')
}

right_order = [
    right_lookup[r[2:]]
    for r in reversed(left_order)
]

mirrored_hemisphere_order = left_order + right_order

print(f'Dim of mirrored hemispheric order: {len(mirrored_hemisphere_order)}')

# sanity checks
print(len(left_order), len(right_order))
print(len(mirrored_hemisphere_order))

assert len(left_order) == 45
assert len(right_order) == 45
assert set(mirrored_hemisphere_order) == set(region_names)

mean_emp_sc_pandas = pd.DataFrame(
    mean_emp_sc,
    columns=region_names,
    index=region_names
)

mirrored_mean_emp_sc = mean_emp_sc_pandas.loc[
    mirrored_hemisphere_order,
    mirrored_hemisphere_order
]

plt.figure(figsize=(8, 8))
plt.imshow(mirrored_mean_emp_sc)
plt.show()

## Change empirical FC to have the same sinus problem

In [ ]:


crit = get_critical_coupling_ou(mirrored_mean_emp_sc)

print(f"Critical coupling: {crit}")

import pandas as pd

couplings = np.linspace(
    0.001,
    0.9999,
    1000
) * crit


# ---------------------------------------------
# Put mean SC and mean FC into labelled DFs
# ---------------------------------------------

mean_emp_sc_pandas = pd.DataFrame(
    mean_emp_sc,
    columns=region_names,
    index=region_names
)

mean_emp_fc_pandas = pd.DataFrame(
    mean_emp_fc,
    columns=region_names,
    index=region_names
)

# ---------------------------------------------
# Apply IDENTICAL ordering
# ---------------------------------------------

mirrored_mean_emp_sc = mean_emp_sc_pandas.loc[
    mirrored_hemisphere_order,
    mirrored_hemisphere_order
]

mirrored_mean_emp_fc = mean_emp_fc_pandas.loc[
    mirrored_hemisphere_order,
    mirrored_hemisphere_order
]

# ---------------------------------------------
# sanity check
# ---------------------------------------------

assert (
    mirrored_mean_emp_sc.columns ==
    mirrored_mean_emp_fc.columns
).all()

assert (
    mirrored_mean_emp_sc.index ==
    mirrored_mean_emp_fc.index
).all()

emp_fc = np.asarray(mirrored_mean_emp_fc, dtype=float).copy()

np.fill_diagonal(emp_fc, 1.0)

# empirical FC has NaN self-correlations
np.fill_diagonal(emp_fc, 1.0)

print("NaNs in empirical FC:", np.isnan(emp_fc).sum())
# should print 0

log = pd.DataFrame(columns=[
    'c',
    'c_over_crit',
    'corr'
])

print("NaNs in empirical FC:", np.isnan(emp_fc).sum())
print("NaNs in SC:", np.isnan(np.asarray(mirrored_mean_emp_sc, dtype=float)).sum())

test_fc, test_cov = ou_fc(
    mirrored_mean_emp_sc,
    coupling=couplings[0],
    sigma=1.0
)

print("NaNs in simulated covariance:", np.isnan(test_cov).sum())
print("NaNs in simulated FC:", np.isnan(test_fc).sum())

print("Cov diagonal min/max:",
      np.diag(test_cov).min(),
      np.diag(test_cov).max())


for coupling in couplings:

    sim_fc, sim_cov = ou_fc(
    mirrored_mean_emp_sc,
    coupling=coupling,
    sigma=1.0
    )

    # --------------------------------------------------------
    # Saggio parameter exploration:
    # retain diagonal during search for best c
    # --------------------------------------------------------

    tri_with_diag = np.tril_indices_from(
        emp_fc,
        k=0
    )

    corr = np.corrcoef(
        sim_fc[tri_with_diag],
        emp_fc[tri_with_diag]
    )[0, 1]

    log.loc[len(log)] = {
        'c': coupling,
        'c_over_crit': coupling / crit,
        'corr': corr
    }



max_idx = log['corr'].idxmax()

best_c = log.loc[max_idx, 'c']
best_corr_sweep = log.loc[max_idx, 'corr']

print(f"Critical coupling: {crit}")
print(f"Best coupling:     {best_c}")
print(f"c / c_crit:        {best_c / crit}")
print(f"Sweep correlation: {best_corr_sweep}")




best_fc, best_cov = ou_fc(
    mirrored_mean_emp_sc,
    coupling=best_c
)

tri_no_diag = np.tril_indices_from(
    emp_fc,
    k=-1
)

final_corr = np.corrcoef(
    best_fc[tri_no_diag],
    emp_fc[tri_no_diag]
)[0, 1]

print(f"Final off-diagonal correlation: {final_corr}")

In [ ]:
plt.title('best FC')
plt.imshow(best_fc)
plt.show()

plt.title('average empirical FC')
plt.imshow(emp_fc)
plt.show()

## Plotting:

In [ ]:
# plt.figure(figsize=(4, 6))

plt.plot(
    log['c'],
    log['corr']
)

plt.axvline(
    crit,
    linestyle='--',
    label=fr'$c_{{critic}}$ = {crit:.3f}'
)

plt.scatter(
    best_c,
    best_corr_sweep,
    color='red',
    zorder=5
)

plt.axvline(
    best_c,
    color='red',
    linestyle='--',
    label=fr'$c_{{best}}$ = {best_c:.3f}'
)

# same visual range as paper
# plt.xlim(0, 1.5)
# plt.ylim(0.35, 0.75)

plt.xlabel('Global coupling parameter c')
plt.ylabel('Correlation between empirical FC and aFC')

plt.legend()
plt.show()

# OLDER CODE: BELOW

## Normalizations:

In [ ]:
import numpy as np
import pandas as pd


sample_sc = ordered_emp_sc_matrix_list[1]

# Matrix magic

def cleanup(sc: pd.DataFrame) -> pd.DataFrame:
    sc = sc.fillna(0)                      # NaN diagonal -> 0
    sc = (sc + sc.T) / 2                   # symmetrize
    np.fill_diagonal(sc.values, 0)         # remove self-loops (in-place)
    return sc

cleaned_sample = cleanup(sample_sc)

fig, axes = plt.subplots(ncols=2, nrows=1, figsize=(10, 4))

axes[0].imshow(sample_sc)
axes[0].set_title("Original")

axes[1].imshow(cleaned_sample)
axes[1].set_title("Cleaned, symmetrized")

plt.tight_layout()
plt.show()

# template:
def f(sc: pd.DataFrame) -> pd.DataFrame:
    return sc

# Spectral radius normalization
def spectral_radius(sc: pd.DataFrame) -> pd.DataFrame:
    sc = sc / np.linalg.eigvalsh(sc.values).max()
    return sc

# Row normalization
def row_norm(sc: pd.DataFrame) -> pd.DataFrame:
    sc = sc.div(sc.sum(axis=1), axis=0)
    return sc

# Max normalization
def max_norm(sc: pd.DataFrame) -> pd.DataFrame:
    sc = sc / sc.values.max()
    return sc

# Symmetric degree normalization
def symmetric_degree_norm(sc: pd.DataFrame) -> pd.DataFrame:
    d = sc.sum(axis=1) ** -0.5
    return sc.mul(d, axis=0).mul(d, axis=1)

# Proportional threshold
def proportional_threshold(sc: pd.DataFrame) -> pd.DataFrame:
    cut = np.quantile(sc.values[np.triu_indices(len(sc), 1)], 0.85)
    sc = sc.where(sc >= cut, 0)
    return sc

# Log transform
def log_transform(sc: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(np.log1p(sc.values), index=sc.index, columns=sc.columns)

# Binarize
def binarize(sc: pd.DataFrame) -> pd.DataFrame:
    sc = (sc > 0).astype(float)
    return sc

# Concensus
def consensus(sc_list: list) -> pd.DataFrame:
    stack = np.stack([s.values for s in sc_list])
    sc = pd.DataFrame(stack.mean(0) * ((stack > 0).mean(0) >= 0.5),
                    index=sc_list[0].index, columns=sc_list[0].columns)
    return sc
    
# Graph laplacian
def graph_laplacian(sc: pd.DataFrame) -> pd.DataFrame:
    L = np.diag(sc.sum(axis=1).to_numpy()) - sc.to_numpy()
    return pd.DataFrame(L, index=sc.index, columns=sc.columns)


# # Largest eigenvalue normalization
# def eigenvalue_norm(df: pd.DataFrame) -> pd.DataFrame:
#     return df


## Older OU model (stationary autoregressive process)

In [ ]:
import numpy as np
# stationary autoregressive process


# The explored parameter’s range goes from c = 0, i.e. uncoupled nodes, to ccritic = 1/maxi λi, where λi are the eigenvalues of W.
def get_critical_coupling(W: pd.DataFrame):
    print(f'def get crit: W shape: {W.shape}')
    eigenvalues, eigenvectors = np.linalg.eig(W)
    lambda_value = max(eigenvalues)
    critical_coupling = 1/lambda_value
    return critical_coupling

def compute_ou_covariance(sigma: float, W: pd.DataFrame, c: float) -> pd.DataFrame:
    colnames = W.columns

    W_np = W.to_numpy(copy=True)

    np.fill_diagonal(W_np, 0)

    n = W_np.shape[0]

    # (-I + cW)
    A = -np.eye(n) + c * W_np

    # C = -(sigma^2 / 2) * (-I + cW)^-1
    C = -(sigma**2 / 2) * np.linalg.inv(A)

    return pd.DataFrame(
        C,
        index=W.index,
        columns=W.columns
    )